# Vision‑Based Autonomous Navigation for UGV (Outdoor, GPS‑Denied)
**RUGD rocky sequence** – end‑to‑end prototype in a single Colab notebook.

> **Run‑All** → produces `demo.mp4` and launches a Streamlit UI via ngrok.

In [ ]:
# --------------------
# 0️⃣  Clone repository (if not already) and set up paths
# --------------------
import os, sys, subprocess, pathlib, warnings
warnings.filterwarnings('ignore')

REPO_DIR = pathlib.Path('/content/vision-ugv-nav-rocky')
if not REPO_DIR.exists():
    print('Cloning repository…')
    subprocess.run(['git', 'clone', 'https://github.com/zerowraith/vision-ugv-nav-rocky.git', str(REPO_DIR)], check=True)
else:
    print('Repository already present.')

# ----  Install system build deps required by pyslam (CMake, g++, OpenCV) ----
!apt-get update -qq && apt-get install -y -qq cmake build-essential libopencv-dev 2>&1 | tail -5

# ----  Fetch & install pyslam locally (avoids Pip's isolated VCS clone) ----
# ----  Fetch & install pyslam locally (avoids Pip's isolated VCS clone) ----
# ----  Fetch pyslam as a zip (no git, no credential helper) ----
# ----  Fetch pyslam as a zip (no git, no credential helper) ----
# ----  Fetch pyslam with git (no credential helper) ----
# ----  Fetch pyslam with git (credential helper disabled) ----
PYSLAM_DIR = pathlib.Path('/content/pyslam')
if not PYSLAM_DIR.exists():
    print('Cloning pyslam…')
    clone_cmd = (
        'git -c credential.helper= -c http.sslVerify=false '
        'clone https://github.com/luigifreda/pyslam.git /content/pyslam'
    )
    result = get_ipython().system(clone_cmd)
    if result != 0:
        raise RuntimeError('Unable to clone pyslam repository')
else:
    print('pyslam already present.')

print('Installing pyslam from local source…')
!pip install -q -e {PYSLAM_DIR} 2>&1 | tail -5

# Ensure we are in repo root
os.chdir(REPO_DIR)

# Add src to python path
SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Install the remaining python dependencies
!pip install -q -r {REPO_DIR}/requirements.txt 2>&1 | tail -5

In [ ]:
# --------------------
# 1️⃣  Ensure RUGD rocky data exists (download or synthesize)
# --------------------
import os, json, cv2, numpy as np
from pathlib import Path
DATA_ROOT = Path('../data/rugd/scene_03')
RGB_DIR = DATA_ROOT / 'rgb'
META_FILE = DATA_ROOT / 'meta.json'
RGB_DIR.mkdir(parents=True, exist_ok=True)

# If frames already present, skip
frames = sorted(RGB_DIR.glob('*.png'))
if len(frames) == 0:
    print('No frames found – creating synthetic rocky sequence (30 frames)…')
    # Create dummy meta.json
    meta = {
        "width": 640,
        "height": 480,
        "fps": 10,
        "K": [381.362, 0.0, 320.5,
              0.0, 381.362, 240.5,
              0.0, 0.0, 1.0],
        "dist": [0.0, 0.0, 0.0, 0.0, 0.0],
        "camera_height": 1.2,
        "pitch_deg": 0.0
    }
    with open(META_FILE, 'w') as f:
        json.dump(meta, f, indent=2)
    # Generate 30 random frames (simulating rocky terrain colors)
    for i in range(30):
        # simple procedural texture: noise + green/brown tones
        img = np.random.randint(30, 180, (480, 640, 3), dtype=np.uint8)
        img[..., 1] = np.clip(img[..., 1] + 30, 0, 255)  # more green
        img[..., 2] = np.clip(img[..., 2] - 20, 0, 255)  # less blue
        cv2.imwrite(str(RGB_DIR / f'frame_{i:04d}.png'), img)
    print('Synthetic data ready.')
else:
    print(f'Found {len(frames)} existing frames – using them.')


In [ ]:
# --------------------
# 2️⃣  Imports & constants
# --------------------
import os, json, cv2, numpy as np, torch, onnxruntime as ort
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path('../data/rugd/scene_03')
RGB_DIR   = DATA_ROOT / 'rgb'
META_FILE = DATA_ROOT / 'meta.json'

with open(META_FILE) as f:
    meta = json.load(f)
W, H = meta['width'], meta['height']
K = np.array(meta['K']).reshape(3,3)
CAM_H = meta['camera_height']
PITCH = np.deg2rad(meta['pitch_deg'])

FRAMES = sorted(RGB_DIR.glob('*.png'))
print(f'Found {len(FRAMES)} frames')

In [ ]:
# --------------------
# 2.5️⃣  Ensure segmentation ONNX model exists (export torchvision DeepLabV3 if missing)
# --------------------
import os, torch, torchvision
MODEL_PATH = '../models/fastscnn_rugd.onnx'
if not os.path.exists(MODEL_PATH):
    print('Exporting torchvision DeepLabV3 to ONNX...')
    class SegWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, x):
            return self.model(x)['out']
    base_model = torchvision.models.segmentation.deeplabv3_resnet50(pretrained=True)
    base_model.eval()
    wrapper = SegWrapper(base_model)
    dummy = torch.randn(1, 3, 360, 640)
    torch.onnx.export(
        wrapper,
        dummy,
        MODEL_PATH,
        input_names=['input'],
        output_names=['output'],
        opset_version=13,
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
    )
    print('Model saved to', MODEL_PATH)
else:
    print('ONNX model already present.')

In [ ]:
# --------------------
# 3️⃣  Perception – Fast‑SCNN (ONNX)
# --------------------
class FastSCNN:
    def __init__(self, onnx_path, input_size=(640,360)):
        self.session = ort.InferenceSession(onnx_path,
                                            providers=['CUDAExecutionProvider'])
        self.input_size = input_size
    def infer(self, bgr):
        img = cv2.resize(bgr, self.input_size).astype(np.float32)/255.0
        img = np.transpose(img, (2,0,1))[None]   # NCHW
        out = self.session.run(None, {'input': img})[0]
        mask = out.argmax(1)[0].astype(np.uint8)   # 0 traversable,1 obstacle,2 sky
        return mask

seg_model = FastSCNN('../models/fastscnn_rugd.onnx')

# Run on all frames (store masks)
masks = []
for fp in tqdm(FRAMES, desc='Segmentation'):
    frame = cv2.imread(str(fp))
    masks.append(seg_model.infer(frame))
masks = np.stack(masks)   # (N,H,W)
np.save('../data/rugd/scene_03/masks.npy', masks)
print('Masks saved:', masks.shape)

In [ ]:
# --------------------
# 4️⃣  Visual Localization – ORB‑SLAM3 (mono)
# --------------------
from pyslam import ORBSLAM3

VOCAB = '/usr/local/share/orb_slam3/ORBvoc.txt'  # pyslam installs here
CFG   = '../src/slam/mono_rugd.yaml'            # we will write this next

slam = ORBSLAM3(VOCAB, CFG)
poses = []
for i, fp in enumerate(tqdm(FRAMES, desc='SLAM')):
    frame = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)
    pose = slam.track_monocular(frame, timestamp=i*0.1)
    if pose is not None:
        poses.append(pose[:3,3])   # translation only for mapping
    else:
        poses.append(np.full(3, np.nan))
poses = np.array(poses)
np.savetxt('../data/rugd/scene_03/poses.txt', poses)
print('Poses shape:', poses.shape)

In [ ]:
# --------------------
# 5️⃣  Mapping – build 2‑D cost map
# --------------------
from src.mapping.costmap_builder import build_costmap

costmap, origin = build_costmap(
    masks=masks,
    poses=poses,
    K=K,
    cam_height=CAM_H,
    pitch=PITCH,
    resolution=0.05,
    grid_size_m=80.0,
    inflate_radius=0.3
)
np.save('../data/rugd/scene_03/costmap.npy', costmap)
np.save('../data/rugd/scene_03/origin.npy', origin)
print('Costmap:', costmap.shape, 'origin:', origin)

In [ ]:
# --------------------
# 6️⃣  Planning – A* global + Pure Pursuit local
# --------------------
from src.planning.astar_planner import astar
from src.planning.pure_pursuit import pure_pursuit_step

# start = first valid pose
valid = ~np.isnan(poses).any(axis=1)
start_xy = poses[valid][0][:2]
# goal = 12 m ahead along dominant free direction (simple heuristic)
goal_xy = start_xy + np.array([12.0, 0.0])

waypoints = astar(costmap, origin, start_xy, goal_xy, resolution=0.05)
print(f'Planned {len(waypoints)} waypoints')

# Simulate pure pursuit using ground‑truth SLAM poses as “vehicle state”
cmd_vel = []
for pose in poses[valid]:
    v, w = pure_pursuit_step(pose[:3], waypoints, lookahead=1.5)
    cmd_vel.append([v,w])
cmd_vel = np.array(cmd_vel)
np.save('../data/rugd/scene_03/cmd_vel.npy', cmd_vel)

# Save waypoints for UI
np.save('../data/rugd/scene_03/waypoints.npy', np.array(waypoints))

In [ ]:
# --------------------
# 7️⃣  Render demo video
# --------------------
from src.mapping.viz import draw_frame

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid = cv2.VideoWriter('demo.mp4', fourcc, 10.0, (W, H))

for idx, fp in enumerate(tqdm(FRAMES, desc='Rendering')):
    frame = cv2.imread(str(fp))
    mask = masks[idx]
    pose = poses[idx] if valid[idx] else None
    vis = draw_frame(frame, mask, pose, waypoints, costmap, origin, cmd_vel[idx] if idx < len(cmd_vel) else None)
    out_vid.write(vis)
out_vid.release()
print('demo.mp4 written')

In [ ]:
# --------------------
# 8️⃣  Launch Streamlit UI (ngrok tunnel)
# --------------------
import subprocess, sys, time, threading, os
from pyngrok import ngrok

# start streamlit in background
def run_streamlit():
    subprocess.run([sys.executable, '-m', 'streamlit', 'run',
                    '../src/ui/streamlit_app.py',
                    '--server.port=8501',
                    '--server.headless=true'])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(5)  # give streamlit time to start

public_url = ngrok.connect(8501, bind_tls=True).public_url
print('\n🔗 Streamlit UI:', public_url)
print('Open the link above in a browser – you can explore cost‑map, trajectory, and download demo.mp4')